# 06 - Gold Pipeline Parcial

Este notebook documenta la capa **Gold**: modelo dimensional tipo **constelación de hechos con copo de nieve parcial**, listo para Power BI y los 6 dashboards de toma de decisión.


```mermaid
graph LR
  S["Silver Curated"] --> G["Gold Dimensional"]
  G --> DM["dim_municipalidad_gold"]
  G --> DU["dim_ubigeo"]
  G --> DT["dim_tiempo"]
  G --> DC["dim_clasificador_ingreso"]
  G --> DE["dim_estado_sismepre"]
  G --> FI["fact_ingresos_mensuales / clasificador"]
  G --> FP["fact_predial_mensual"]
  G --> FR["facts RENAMU"]
  G --> BI["6 dashboards Power BI"]
```


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('notebook-gold-evidence').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
gold = Path('/home/jovyan/work/data/gold')
tables = sorted([p.name for p in gold.iterdir() if p.is_dir() and list(p.rglob('*.parquet'))])
summary = []
for table in tables:
    df = spark.read.parquet(str(gold / table))
    kind = 'dimension' if table.startswith('dim_') else ('fact' if table.startswith('fact_') else 'mart')
    summary.append((table, kind, df.count(), len(df.columns)))
spark.createDataFrame(summary, ['tabla_gold', 'tipo', 'registros', 'columnas']).show(100, truncate=False)


## Modelo Gold

Dimensiones principales: `dim_municipalidad_gold`, `dim_ubigeo`, `dim_tiempo`, `dim_clasificador_ingreso`, `dim_estado_sismepre`, `dim_formulario_sismepre`, `dim_pregunta_sismepre`.

Facts principales: `fact_ingresos_mensuales`, `fact_ingresos_clasificador`, `fact_predial_mensual`, `fact_sismepre_cumplimiento`, `fact_sismepre_respuestas_resumen`, `fact_renamu_gestion_tributaria`, `fact_renamu_software_at`.

`mart_dashboard_municipal` es una vista de consumo para acelerar dashboards, no reemplaza el modelo dimensional.


In [ ]:
municipios = spark.read.parquet(str(gold / 'dim_municipalidad_gold'))
municipios.groupBy('categoria_municipalidad', 'categoria_match_status').count().orderBy('categoria_municipalidad', 'categoria_match_status').show(50, truncate=False)


## Los 6 Dashboards

1. Recaudación Municipal vs Capacidad Tributaria: SIAF + RENAMU + Categorías.
2. Recaudación por Clasificador de Ingreso: SIAF + clasificador + Categorías.
3. Predial vs Efectividad: SISMEPRE + municipalidades + Categorías.
4. Distribución de Efectividad Predial: SISMEPRE + Categorías.
5. Software Tributario Municipal: RENAMU + Categorías.
6. Priorización de Municipalidades: SIAF + SISMEPRE + RENAMU + Categorías.


In [ ]:
quality = spark.read.parquet(str(gold / 'fact_calidad_datos'))
quality.groupBy('layer', 'status').count().orderBy('layer', 'status').show(50, truncate=False)
